# 🔬 Critical Experiments for Reviewer Response

## Three experiments:
1. **Fine-tuned Baseline** — Same Phase 2 protocol on baseline WITHOUT MR-ResBlocks
2. **Conv-Only (3 blocks)** — MR-ResBlocks only after Conv layers (no pre/post LSTM)
3. **Parameter Count Verification** — Exact counts for all model variants

## Instructions:
- Make sure `baseline_cnn_bilstm_best.pt` path is correct in Cell 1
- Run ALL cells in order (1 → 7)
- Screenshot the FINAL SUMMARY at the end

## Expected runtime: ~1-2 hours total (2 training runs)

In [1]:
# ============================================================
# STEP 0: Imports + Settings + Seed
# ============================================================
import os, copy, random, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# ---------- SETTINGS ----------
MANIFEST_CSV = r"C:\Users\PC\Desktop\courses\Conferences\Sevile\test\dataset\manifest_pairs.csv"
ALL_SUBJECTS = ["SUB1","SUB2","SUB3","SUB4","SUB5","SUB6","SUB7","SUB8","SUB9","SUB10"]

TARGET_CH  = 127
T_FIXED    = 1349
MEL_BINS   = 80
MEL_FRAMES = 259

BATCH_SIZE   = 4
PATIENCE     = 8
WEIGHT_DECAY = 1e-4

# Phase 1: Train baseline CNN-BiLSTM
EPOCHS_BASELINE = 60
LR_BASELINE     = 1e-4

# Phase 2: Fine-tune hybrid model
EPOCHS_HYBRID   = 60
LR_HYBRID       = 5e-5   # أقل لأننا نبدأ من checkpoint جيد

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- SEED ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f"Device: {device}")

Device: cpu


In [2]:
# ============================================================
# STEP 1: Utility Functions (Time crop/pad)
# ============================================================
def pad_or_crop_time(x, T_fixed=T_FIXED):
    C, T = x.shape
    if T == T_fixed: return x
    if T > T_fixed:  return x[:, :T_fixed]
    out = np.zeros((C, T_fixed), dtype=x.dtype)
    out[:, :T] = x
    return out

def center_crop_time(x, T_fixed=T_FIXED):
    C, T = x.shape
    if T <= T_fixed: return pad_or_crop_time(x, T_fixed)
    start = (T - T_fixed) // 2
    return x[:, start:start + T_fixed]

def random_crop_time(x, T_fixed=T_FIXED):
    C, T = x.shape
    if T <= T_fixed: return pad_or_crop_time(x, T_fixed)
    start = np.random.randint(0, T - T_fixed + 1)
    return x[:, start:start + T_fixed]

print("✅ Utility functions ready")

✅ Utility functions ready


In [3]:
# ============================================================
# STEP 2: Per-Subject EEG Statistics
# ============================================================
def compute_eeg_stats_train(manifest_csv, subject):
    df = pd.read_csv(manifest_csv)
    df = df[(df["subject"] == subject) & (df["split"] == "train")].reset_index(drop=True)
    sum_c = sumsq_c = None
    count_t = 0
    for p in df["eeg_path"].tolist():
        x = np.load(p)
        if x.shape[0] > x.shape[1]: x = x.T
        x = x.astype(np.float64)
        if sum_c is None:
            sum_c   = x.sum(axis=1)
            sumsq_c = (x**2).sum(axis=1)
        else:
            sum_c   += x.sum(axis=1)
            sumsq_c += (x**2).sum(axis=1)
        count_t += x.shape[1]
    mean = (sum_c / count_t).astype(np.float32)
    var  = (sumsq_c / count_t - mean.astype(np.float64)**2).astype(np.float32)
    std  = np.sqrt(np.maximum(var, 1e-8)).astype(np.float32)
    return mean, std

eeg_stats = {s: compute_eeg_stats_train(MANIFEST_CSV, s) for s in ALL_SUBJECTS}
print("✅ Computed stats for", len(eeg_stats), "subjects")

✅ Computed stats for 10 subjects


In [4]:
# ============================================================
# STEP 3: Dataset
# ============================================================
class MultiSubjectMaskedDataset(Dataset):
    def __init__(self, manifest_csv, split, subjects, eeg_stats):
        self.df = pd.read_csv(manifest_csv)
        self.df = self.df[(self.df["split"] == split) &
                          (self.df["subject"].isin(subjects))].reset_index(drop=True)
        self.split     = split
        self.subjects  = list(subjects)
        self.sub2idx   = {s: i for i, s in enumerate(self.subjects)}
        self.eeg_stats = eeg_stats

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        subj = row["subject"]
        eeg  = np.load(row["eeg_path"])
        if eeg.shape[0] > eeg.shape[1]: eeg = eeg.T
        eeg = eeg.astype(np.float32)
        mel = np.load(row["mel_path"]).astype(np.float32)

        mean, std = self.eeg_stats[subj]
        eeg = (eeg - mean[:, None]) / (std[:, None] + 1e-8)

        C, T = eeg.shape
        eeg_pad = np.zeros((TARGET_CH, T), dtype=np.float32)
        eeg_pad[:C, :] = eeg
        mask = np.zeros((TARGET_CH, 1), dtype=np.float32)
        mask[:C, 0] = 1.0

        if self.split == "train":
            eeg_pad = random_crop_time(eeg_pad)
        else:
            eeg_pad = center_crop_time(eeg_pad)

        sid = self.sub2idx[subj]
        return (torch.from_numpy(eeg_pad), torch.from_numpy(mel),
                torch.from_numpy(mask), torch.tensor(sid, dtype=torch.long))

print("✅ Dataset class defined")

✅ Dataset class defined


In [5]:
# ============================================================
# STEP 4: DataLoaders
# ============================================================
train_set = MultiSubjectMaskedDataset(MANIFEST_CSV, "train", ALL_SUBJECTS, eeg_stats)
val_set   = MultiSubjectMaskedDataset(MANIFEST_CSV, "val",   ALL_SUBJECTS, eeg_stats)
test_set  = MultiSubjectMaskedDataset(MANIFEST_CSV, "test",  ALL_SUBJECTS, eeg_stats)

sub_counts = train_set.df["subject"].value_counts().to_dict()
weights = [1.0 / sub_counts[s] for s in train_set.df["subject"].tolist()]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,  num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,  num_workers=0)

print(f"✅ Loaders: Train={len(train_set)}, Val={len(val_set)}, Test={len(test_set)}")

✅ Loaders: Train=796, Val=99, Test=101


In [6]:
# ============================================================
# STEP 5-A: Baseline Model (CNN-BiLSTM) — نفس الأصلي بالضبط
# ============================================================
class CNNBiLSTM_SubjectEmbed(nn.Module):
    """Baseline model - same as original."""
    def __init__(self, n_subjects=10, embed_dim=32, hidden=256):
        super().__init__()
        self.sub_embed = nn.Embedding(n_subjects, embed_dim)
        self.encoder = nn.Sequential(
            nn.Conv1d(TARGET_CH, 128, kernel_size=7, padding=3), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 256, kernel_size=7, padding=3),       nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256, 256, kernel_size=7, padding=3),       nn.ReLU(), nn.MaxPool1d(2),
        )
        self.time_pool = nn.AdaptiveAvgPool1d(MEL_FRAMES)
        self.bilstm = nn.LSTM(input_size=256 + embed_dim, hidden_size=hidden,
                              num_layers=1, batch_first=True, bidirectional=True)
        self.proj = nn.Linear(hidden * 2, MEL_BINS)

    def forward(self, x, mask, sid):
        x = x * mask
        z = self.encoder(x)
        z = self.time_pool(z)
        z = z.permute(0, 2, 1)
        e = self.sub_embed(sid).unsqueeze(1).repeat(1, z.size(1), 1)
        z = torch.cat([z, e], dim=-1)
        z, _ = self.bilstm(z)
        z = self.proj(z)
        z = z.permute(0, 2, 1)
        return z

print("✅ Baseline CNN-BiLSTM defined")

# ============================================================
# STEP 5-B: NeuroTalk Multi-Receptive ResBlock
# ============================================================
class MultiReceptiveResBlock(nn.Module):
    """
    Core component from NeuroTalk (Lee et al., AAAI 2023).
    Parallel Conv1d with kernels 3,5,7 → merge → residual.
    Captures multi-scale frequency characteristics.
    """
    def __init__(self, channels):
        super().__init__()
        self.conv_k3 = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels), nn.LeakyReLU(0.2))
        self.conv_k5 = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(channels), nn.LeakyReLU(0.2))
        self.conv_k7 = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=7, padding=3),
            nn.BatchNorm1d(channels), nn.LeakyReLU(0.2))
        self.merge = nn.Sequential(
            nn.Conv1d(channels * 3, channels, kernel_size=1),
            nn.BatchNorm1d(channels), nn.LeakyReLU(0.2))
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        h = torch.cat([self.conv_k3(x), self.conv_k5(x), self.conv_k7(x)], dim=1)
        return x + self.dropout(self.merge(h))

print("✅ MultiReceptiveResBlock defined")

# ============================================================
# STEP 5-C: Hybrid Model (CNN-BiLSTM + NeuroTalk ResBlocks)
# ============================================================
#
# الفكرة: نأخذ CNN-BiLSTM الأصلي ونُدخل NeuroTalk ResBlocks
# في 3 مواقع استراتيجية:
#   1. بعد كل طبقة Conv في الـ encoder
#   2. قبل الـ BiLSTM (refinement)
#   3. بعد الـ BiLSTM (post-processing)
#
# الـ CNN encoder layers تبقى نفسها → يمكن نقل الـ weights
# الـ BiLSTM يبقى نفسه → يمكن نقل الـ weights
# فقط NeuroTalk ResBlocks جديدة → تتدرب من الصفر

class HybridNeuroTalk(nn.Module):
    """
    Hybrid: Original CNN-BiLSTM backbone + NeuroTalk Multi-Receptive ResBlocks.
    
    Architecture:
    EEG → [Conv1→MRResBlock] → [Conv2→MRResBlock] → [Conv3→MRResBlock]
        → Pool → [MRResBlock_pre] → BiLSTM → [MRResBlock_post] → Mel
    
    Transfer: Conv1-3 + BiLSTM weights loaded from baseline checkpoint.
    """
    def __init__(self, n_subjects=10, embed_dim=32, hidden=256):
        super().__init__()
        
        self.sub_embed = nn.Embedding(n_subjects, embed_dim)
        
        # ---- Original CNN Encoder (weights transferable) ----
        self.conv1 = nn.Sequential(
            nn.Conv1d(TARGET_CH, 128, kernel_size=7, padding=3),
            nn.ReLU(), nn.MaxPool1d(2))
        self.conv2 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=7, padding=3),
            nn.ReLU(), nn.MaxPool1d(2))
        self.conv3 = nn.Sequential(
            nn.Conv1d(256, 256, kernel_size=7, padding=3),
            nn.ReLU(), nn.MaxPool1d(2))
        
        # ---- NeuroTalk Multi-Receptive ResBlocks (جديدة) ----
        self.mr_res1 = MultiReceptiveResBlock(128)   # بعد conv1
        self.mr_res2 = MultiReceptiveResBlock(256)   # بعد conv2
        self.mr_res3 = MultiReceptiveResBlock(256)   # بعد conv3
        self.mr_pre_lstm = MultiReceptiveResBlock(256)  # قبل BiLSTM
        
        self.time_pool = nn.AdaptiveAvgPool1d(MEL_FRAMES)
        
        # ---- Original BiLSTM (weights transferable) ----
        self.bilstm = nn.LSTM(
            input_size=256 + embed_dim,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        # ---- Post-LSTM NeuroTalk refinement (جديد) ----
        self.post_lstm_proj = nn.Linear(hidden * 2, 256)
        self.mr_post_lstm  = MultiReceptiveResBlock(256)
        
        # ---- Final projection ----
        self.proj = nn.Linear(256, MEL_BINS)
    
    def forward(self, x, mask, sid):
        # x: (B, 127, T), mask: (B, 127, 1)
        x = x * mask
        
        # Encoder with interleaved NeuroTalk ResBlocks
        z = self.conv1(x)            # (B, 128, T/2)
        z = self.mr_res1(z)          # ← NeuroTalk: multi-receptive features
        
        z = self.conv2(z)            # (B, 256, T/4)
        z = self.mr_res2(z)          # ← NeuroTalk
        
        z = self.conv3(z)            # (B, 256, T/8)
        z = self.mr_res3(z)          # ← NeuroTalk
        z = self.mr_pre_lstm(z)      # ← NeuroTalk: pre-LSTM refinement
        
        z = self.time_pool(z)        # (B, 256, 259)
        z = z.permute(0, 2, 1)       # (B, 259, 256)
        
        # Subject embedding
        e = self.sub_embed(sid)      # (B, embed_dim)
        e = e.unsqueeze(1).repeat(1, z.size(1), 1)  # (B, 259, embed_dim)
        z = torch.cat([z, e], dim=-1)  # (B, 259, 256+embed_dim)
        
        # BiLSTM
        z, _ = self.bilstm(z)        # (B, 259, 512)
        
        # Post-LSTM NeuroTalk refinement
        z = self.post_lstm_proj(z)   # (B, 259, 256)
        z = z.permute(0, 2, 1)       # (B, 256, 259)
        z = self.mr_post_lstm(z)     # ← NeuroTalk: post-LSTM refinement
        z = z.permute(0, 2, 1)       # (B, 259, 256)
        
        # Final projection
        z = self.proj(z)             # (B, 259, 80)
        z = z.permute(0, 2, 1)       # (B, 80, 259)
        return z


# ---- إحصائيات الموديل ----
_m = HybridNeuroTalk(n_subjects=10)
total    = sum(p.numel() for p in _m.parameters())
new_only = sum(p.numel() for n, p in _m.named_parameters()
               if 'mr_res' in n or 'mr_pre' in n or 'mr_post' in n or 'post_lstm_proj' in n)
old_transferable = total - new_only
print(f"✅ Hybrid NeuroTalk Model")
print(f"   Total params:        {total:,}")
print(f"   Transferable (old):  {old_transferable:,}")
print(f"   New (NeuroTalk):     {new_only:,}")
del _m

✅ Baseline CNN-BiLSTM defined
✅ MultiReceptiveResBlock defined
✅ Hybrid NeuroTalk Model
   Total params:        7,100,304
   Transferable (old):  1,941,648
   New (NeuroTalk):     5,158,656


In [7]:
# ============================================================
# 🔬 ALL CRITICAL EXPERIMENTS
# ============================================================
# Experiment 1: Fine-tuned Baseline (NO MR-ResBlocks)
# Experiment 2: Conv-Only Hybrid (3 MR-ResBlocks, no pre/post LSTM)
# Experiment 3: Parameter count verification
# ============================================================

CKPT_BASELINE = "baseline_cnn_bilstm_best.pt"
# ^^^ تأكدي ان المسار صحيح. إذا مختلف، غيّريه هنا:
# CKPT_BASELINE = r"C:\Users\PC\Desktop\courses\...\baseline_cnn_bilstm_best.pt"

criterion = nn.MSELoss()

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def run_epoch_exp(model, loader, optimizer=None, train=True):
    model.train() if train else model.eval()
    total, n = 0.0, 0
    for xb, yb, mask, sid in loader:
        xb, yb = xb.to(device), yb.to(device)
        mask, sid = mask.to(device), sid.to(device)
        yp = model(xb, mask, sid)
        loss = criterion(yp, yb)
        if train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total += loss.item() * xb.size(0)
        n += xb.size(0)
    return total / max(n, 1)


def train_experiment(model, ckpt_name, exp_name, use_differential_lr=True):
    """Train with same protocol as Phase 2."""

    if use_differential_lr:
        transferred_p, new_p = [], []
        for name, param in model.named_parameters():
            if any(k in name for k in ['mr_res','mr_pre','mr_post','post_lstm_proj']):
                new_p.append(param)
            else:
                transferred_p.append(param)

        if len(new_p) > 0:
            optimizer = torch.optim.Adam([
                {"params": transferred_p, "lr": LR_HYBRID * 0.1},
                {"params": new_p, "lr": LR_HYBRID},
            ], weight_decay=WEIGHT_DECAY)
        else:
            # No new params (fine-tuned baseline) — all at low LR
            optimizer = torch.optim.Adam(
                transferred_p, lr=LR_HYBRID * 0.1, weight_decay=WEIGHT_DECAY)
    else:
        optimizer = torch.optim.Adam(
            model.parameters(), lr=LR_HYBRID, weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2)

    print(f"\n{'='*60}")
    print(f"TRAINING: {exp_name}")
    print(f"{'='*60}")

    best_val, best_ep, pat = float("inf"), -1, 0
    for epoch in range(1, EPOCHS_HYBRID + 1):
        tr = run_epoch_exp(model, train_loader, optimizer, train=True)
        va = run_epoch_exp(model, val_loader, train=False)
        scheduler.step(va)
        print(f"Epoch {epoch:02d} | Train {tr:.4f} | Val {va:.4f}")
        if va < best_val:
            best_val, best_ep, pat = va, epoch, 0
            torch.save(model.state_dict(), ckpt_name)
            print(f"  ✅ Saved")
        else:
            pat += 1
            if pat >= PATIENCE:
                print(f"  🛑 Early stop at epoch {epoch}")
                break

    print(f"\n✅ {exp_name} done. Best epoch={best_ep}, Val={best_val:.4f}")
    model.load_state_dict(torch.load(ckpt_name, map_location=device))
    return model


def evaluate_experiment(model, loader, name):
    model.eval()
    from collections import defaultdict
    results = defaultdict(lambda: {"p":[],"t":[]})
    all_p, all_t = [], []
    with torch.no_grad():
        for xb, yb, mask, sid in loader:
            xb, yb = xb.to(device), yb.to(device)
            mask, sid = mask.to(device), sid.to(device)
            yp = model(xb, mask, sid)
            for i in range(xb.size(0)):
                s = ALL_SUBJECTS[sid[i].item()]
                pf = yp[i].cpu().numpy().flatten()
                tf = yb[i].cpu().numpy().flatten()
                results[s]["p"].append(pf)
                results[s]["t"].append(tf)
                all_p.append(pf)
                all_t.append(tf)
    ap, at = np.concatenate(all_p), np.concatenate(all_t)
    gpcc = np.corrcoef(ap, at)[0,1]
    gmse = np.mean((ap - at)**2)
    gmae = np.mean(np.abs(ap - at))
    print(f"\n{'='*55}")
    print(f"RESULTS: {name}")
    print(f"  PCC = {gpcc:.4f} | MSE = {gmse:.4f} | MAE = {gmae:.4f}")
    print(f"{'='*55}")
    spccs = {}
    for s in ALL_SUBJECTS:
        if s in results:
            sp = np.concatenate(results[s]["p"])
            st = np.concatenate(results[s]["t"])
            spccs[s] = np.corrcoef(sp, st)[0,1]
            smse = np.mean((sp - st)**2)
            print(f"  {s}: PCC={spccs[s]:.3f}, MSE={smse:.3f}")
    return gpcc, gmse, gmae, spccs


# ============================================================
# CONV-ONLY HYBRID (3 MR-ResBlocks — no pre/post LSTM)
# ============================================================
class HybridNeuroTalk_ConvOnly(nn.Module):
    """
    3 MR-ResBlocks: only after Conv layers.
    NO pre-LSTM block, NO post-LSTM block.
    Direct BiLSTM → projection like baseline.
    """
    def __init__(self, n_subjects=10, embed_dim=32, hidden=256):
        super().__init__()
        self.sub_embed = nn.Embedding(n_subjects, embed_dim)
        self.conv1 = nn.Sequential(
            nn.Conv1d(TARGET_CH, 128, kernel_size=7, padding=3),
            nn.ReLU(), nn.MaxPool1d(2))
        self.conv2 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=7, padding=3),
            nn.ReLU(), nn.MaxPool1d(2))
        self.conv3 = nn.Sequential(
            nn.Conv1d(256, 256, kernel_size=7, padding=3),
            nn.ReLU(), nn.MaxPool1d(2))

        # Only 3 MR-ResBlocks (after each Conv)
        self.mr_res1 = MultiReceptiveResBlock(128)
        self.mr_res2 = MultiReceptiveResBlock(256)
        self.mr_res3 = MultiReceptiveResBlock(256)

        self.time_pool = nn.AdaptiveAvgPool1d(MEL_FRAMES)
        self.bilstm = nn.LSTM(input_size=256+embed_dim, hidden_size=hidden,
                              num_layers=1, batch_first=True, bidirectional=True)
        self.proj = nn.Linear(hidden * 2, MEL_BINS)  # Direct 512→80

    def forward(self, x, mask, sid):
        x = x * mask
        z = self.conv1(x);  z = self.mr_res1(z)
        z = self.conv2(z);  z = self.mr_res2(z)
        z = self.conv3(z);  z = self.mr_res3(z)
        # NO pre-LSTM, NO post-LSTM
        z = self.time_pool(z).permute(0, 2, 1)
        e = self.sub_embed(sid).unsqueeze(1).repeat(1, z.size(1), 1)
        z = torch.cat([z, e], dim=-1)
        z, _ = self.bilstm(z)
        z = self.proj(z).permute(0, 2, 1)
        return z


def transfer_baseline_weights(model, name):
    """Transfer baseline weights to any model with conv1/2/3 + bilstm."""
    baseline_sd = torch.load(CKPT_BASELINE, map_location=device)
    mapping = {
        "sub_embed.weight": "sub_embed.weight",
        "encoder.0.weight": "conv1.0.weight", "encoder.0.bias": "conv1.0.bias",
        "encoder.3.weight": "conv2.0.weight", "encoder.3.bias": "conv2.0.bias",
        "encoder.6.weight": "conv3.0.weight", "encoder.6.bias": "conv3.0.bias",
        "bilstm.weight_ih_l0": "bilstm.weight_ih_l0",
        "bilstm.weight_hh_l0": "bilstm.weight_hh_l0",
        "bilstm.bias_ih_l0": "bilstm.bias_ih_l0",
        "bilstm.bias_hh_l0": "bilstm.bias_hh_l0",
        "bilstm.weight_ih_l0_reverse": "bilstm.weight_ih_l0_reverse",
        "bilstm.weight_hh_l0_reverse": "bilstm.weight_hh_l0_reverse",
        "bilstm.bias_ih_l0_reverse": "bilstm.bias_ih_l0_reverse",
        "bilstm.bias_hh_l0_reverse": "bilstm.bias_hh_l0_reverse",
    }
    sd = model.state_dict()
    t = 0
    for sk, dk in mapping.items():
        if sk in baseline_sd and dk in sd:
            if baseline_sd[sk].shape == sd[dk].shape:
                sd[dk] = baseline_sd[sk]; t += 1
    model.load_state_dict(sd)
    print(f"✅ [{name}] Transferred {t}/{len(mapping)} weights")
    return model


# ============================================================
# EXPERIMENT 1: PARAMETER COUNTS (runs instantly)
# ============================================================
print("\n" + "📊"*25)
print("EXPERIMENT 1: PARAMETER COUNTS")
print("📊"*25)

# Baseline
_b = CNNBiLSTM_SubjectEmbed(n_subjects=10, embed_dim=32, hidden=256)
baseline_params = sum(p.numel() for p in _b.parameters())
print(f"\n  Baseline CNN-BiLSTM:        {baseline_params:>10,} params")

# Full Hybrid (5 blocks)
_h = HybridNeuroTalk(n_subjects=10, embed_dim=32, hidden=256)
hybrid_params = sum(p.numel() for p in _h.parameters())
hybrid_new = sum(p.numel() for n, p in _h.named_parameters()
                 if any(k in n for k in ['mr_res','mr_pre','mr_post','post_lstm_proj']))
print(f"  Full Hybrid (5 blocks):     {hybrid_params:>10,} params")
print(f"    — Transferred from base:  {hybrid_params - hybrid_new:>10,}")
print(f"    — New (MR-ResBlocks etc):  {hybrid_new:>10,}")

# Conv-Only (3 blocks)
_c = HybridNeuroTalk_ConvOnly(n_subjects=10, embed_dim=32, hidden=256)
conv_params = sum(p.numel() for p in _c.parameters())
print(f"  Conv-Only (3 blocks):       {conv_params:>10,} params")

# No Post-LSTM (4 blocks) — from previous ablation
# We define it here just for param count
class _NoPost(nn.Module):
    def __init__(self):
        super().__init__()
        self.sub_embed = nn.Embedding(10, 32)
        self.conv1 = nn.Sequential(nn.Conv1d(127, 128, 7, padding=3), nn.ReLU(), nn.MaxPool1d(2))
        self.conv2 = nn.Sequential(nn.Conv1d(128, 256, 7, padding=3), nn.ReLU(), nn.MaxPool1d(2))
        self.conv3 = nn.Sequential(nn.Conv1d(256, 256, 7, padding=3), nn.ReLU(), nn.MaxPool1d(2))
        self.mr_res1 = MultiReceptiveResBlock(128)
        self.mr_res2 = MultiReceptiveResBlock(256)
        self.mr_res3 = MultiReceptiveResBlock(256)
        self.mr_pre_lstm = MultiReceptiveResBlock(256)
        self.bilstm = nn.LSTM(288, 256, 1, batch_first=True, bidirectional=True)
        self.proj = nn.Linear(512, 80)
nopost_params = sum(p.numel() for p in _NoPost().parameters())
print(f"  No Post-LSTM (4 blocks):    {nopost_params:>10,} params")

class _NoPre(nn.Module):
    def __init__(self):
        super().__init__()
        self.sub_embed = nn.Embedding(10, 32)
        self.conv1 = nn.Sequential(nn.Conv1d(127, 128, 7, padding=3), nn.ReLU(), nn.MaxPool1d(2))
        self.conv2 = nn.Sequential(nn.Conv1d(128, 256, 7, padding=3), nn.ReLU(), nn.MaxPool1d(2))
        self.conv3 = nn.Sequential(nn.Conv1d(256, 256, 7, padding=3), nn.ReLU(), nn.MaxPool1d(2))
        self.mr_res1 = MultiReceptiveResBlock(128)
        self.mr_res2 = MultiReceptiveResBlock(256)
        self.mr_res3 = MultiReceptiveResBlock(256)
        self.bilstm = nn.LSTM(288, 256, 1, batch_first=True, bidirectional=True)
        self.post_lstm_proj = nn.Linear(512, 256)
        self.mr_post_lstm = MultiReceptiveResBlock(256)
        self.proj = nn.Linear(256, 80)
nopre_params = sum(p.numel() for p in _NoPre().parameters())
print(f"  No Pre-LSTM (4 blocks):     {nopre_params:>10,} params")

del _b, _h, _c

print(f"\n  📋 SUMMARY:")
print(f"  {'Model':<30} {'Params':>12} {'PCC':>8}")
print(f"  {'-'*55}")
print(f"  {'Baseline':<30} {baseline_params:>12,} {'0.2878':>8}")
print(f"  {'Fine-tuned Baseline':<30} {baseline_params:>12,} {'?????':>8}")
print(f"  {'Conv-Only (3 blocks)':<30} {conv_params:>12,} {'?????':>8}")
print(f"  {'No Post-LSTM (4 blocks)':<30} {nopost_params:>12,} {'0.2814':>8}")
print(f"  {'No Pre-LSTM (4 blocks)':<30} {nopre_params:>12,} {'0.1894':>8}")
print(f"  {'Full Hybrid (5 blocks)':<30} {hybrid_params:>12,} {'0.3998':>8}")


# ============================================================
# EXPERIMENT 2: FINE-TUNED BASELINE (الأهم!)
# ============================================================
# نأخذ الـ baseline المُدرَّب ونطبّق عليه نفس Phase 2
# بدون إضافة أي MR-ResBlocks
# هذا يجيب على السؤال: هل التحسن من الـ architecture ولا من التدريب الإضافي؟
# ============================================================
print("\n\n" + "🔴"*25)
print("EXPERIMENT 2: FINE-TUNED BASELINE (CRITICAL!)")
print("Same Phase 2 protocol, NO MR-ResBlocks")
print("🔴"*25)

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# Load the trained baseline
ft_baseline = CNNBiLSTM_SubjectEmbed(
    n_subjects=len(ALL_SUBJECTS), embed_dim=32, hidden=256).to(device)
ft_baseline.load_state_dict(
    torch.load(CKPT_BASELINE, map_location=device))
print("✅ Loaded baseline checkpoint")

# Apply SAME Phase 2 training protocol
# All params at the LOW learning rate (same as transferred params in hybrid)
ft_optimizer = torch.optim.Adam(
    ft_baseline.parameters(),
    lr=LR_HYBRID * 0.1,  # 5e-6, same as transferred layers in hybrid
    weight_decay=WEIGHT_DECAY
)
ft_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    ft_optimizer, mode="min", factor=0.5, patience=2)

print(f"\nTraining baseline with Phase 2 protocol:")
print(f"  LR = {LR_HYBRID * 0.1:.1e} (same as transferred layers in hybrid)")
print(f"  Max epochs = {EPOCHS_HYBRID}")
print(f"  Early stopping patience = {PATIENCE}")

CKPT_FT = "finetuned_baseline_best.pt"
best_val_ft, best_ep_ft, pat_ft = float("inf"), -1, 0

print(f"\n{'='*60}")
print("PHASE 2 on BASELINE (no architecture changes)")
print(f"{'='*60}")

for epoch in range(1, EPOCHS_HYBRID + 1):
    tr = run_epoch_exp(ft_baseline, train_loader, ft_optimizer, train=True)
    va = run_epoch_exp(ft_baseline, val_loader, train=False)
    ft_scheduler.step(va)
    lr_now = ft_optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch:02d} | Train {tr:.4f} | Val {va:.4f} | LR {lr_now:.1e}")
    if va < best_val_ft:
        best_val_ft, best_ep_ft, pat_ft = va, epoch, 0
        torch.save(ft_baseline.state_dict(), CKPT_FT)
        print(f"  ✅ Saved")
    else:
        pat_ft += 1
        if pat_ft >= PATIENCE:
            print(f"  🛑 Early stop at epoch {epoch}")
            break

print(f"\n✅ Fine-tuned baseline done. Best epoch={best_ep_ft}, Val={best_val_ft:.4f}")
ft_baseline.load_state_dict(torch.load(CKPT_FT, map_location=device))
pcc_FT, mse_FT, mae_FT, subj_FT = evaluate_experiment(
    ft_baseline, test_loader, "Fine-tuned Baseline")


# ============================================================
# EXPERIMENT 3: CONV-ONLY (3 blocks)
# ============================================================
print("\n\n" + "🟡"*25)
print("EXPERIMENT 3: CONV-ONLY (3 MR-ResBlocks)")
print("No pre-LSTM, no post-LSTM")
print("🟡"*25)

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

model_C = HybridNeuroTalk_ConvOnly(
    n_subjects=len(ALL_SUBJECTS), embed_dim=32, hidden=256).to(device)
model_C = transfer_baseline_weights(model_C, "Conv-Only")
model_C = train_experiment(model_C, "ablation_conv_only.pt", "Conv-Only (3 blocks)")
pcc_C, mse_C, mae_C, subj_C = evaluate_experiment(
    model_C, test_loader, "Conv-Only (3 blocks)")


# ============================================================
# 📊 FINAL COMPREHENSIVE SUMMARY
# ============================================================
print("\n\n" + "="*70)
print("📊 COMPREHENSIVE ABLATION + FINE-TUNING SUMMARY")
print("="*70)

print(f"\n{'Model':<35} {'Params':>10} {'PCC':>8} {'MSE':>8} {'Blocks':>7}")
print("-"*72)
print(f"{'Baseline CNN-BiLSTM':<35} {baseline_params:>10,} {'0.2878':>8} {'4.5667':>8} {'0':>7}")
print(f"{'Fine-tuned Baseline ⭐':<35} {baseline_params:>10,} {pcc_FT:>8.4f} {mse_FT:>8.4f} {'0':>7}")
print(f"{'Conv-Only (3 blocks)':<35} {conv_params:>10,} {pcc_C:>8.4f} {mse_C:>8.4f} {'3':>7}")
print(f"{'No Post-LSTM (4 blocks)':<35} {nopost_params:>10,} {'0.2814':>8} {'4.5844':>8} {'4':>7}")
print(f"{'No Pre-LSTM (4 blocks)':<35} {nopre_params:>10,} {'0.1894':>8} {'5.5031':>8} {'4':>7}")
print(f"{'Full Hybrid (5 blocks) ⭐':<35} {hybrid_params:>10,} {'0.3998':>8} {'4.1936':>8} {'5':>7}")
print("-"*72)

print(f"\n🔑 KEY QUESTION: Does fine-tuning alone explain the improvement?")
print(f"   Baseline PCC:            0.2878")
print(f"   Fine-tuned Baseline PCC: {pcc_FT:.4f}")
ft_delta = pcc_FT - 0.2878
print(f"   Delta from fine-tuning:  {ft_delta:+.4f}")
print(f"   Full Hybrid PCC:         0.3998")
hybrid_delta = 0.3998 - 0.2878
arch_delta = 0.3998 - pcc_FT
print(f"   Total improvement:       {hybrid_delta:+.4f}")
print(f"   Attributed to fine-tuning: {ft_delta:+.4f} ({100*ft_delta/hybrid_delta:.1f}%)")
print(f"   Attributed to MR-ResBlocks: {arch_delta:+.4f} ({100*arch_delta/hybrid_delta:.1f}%)")

if pcc_FT < 0.30:
    print(f"\n   ✅ GOOD NEWS: Fine-tuning barely helps → MR-ResBlocks drive the gain!")
elif pcc_FT < 0.35:
    print(f"\n   ⚠️ MIXED: Some gain from fine-tuning, but MR-ResBlocks still needed")
else:
    print(f"\n   ❌ CONCERN: Fine-tuning alone explains much of the improvement")

# Per-subject comparison
print(f"\n\nPER-SUBJECT COMPARISON:")
print(f"{'Subject':<8} {'Baseline':>10} {'FT-Base':>10} {'Hybrid':>10} {'Δ(FT)':>8} {'Δ(Hybrid)':>10}")
print("-"*60)
baseline_pccs = {"SUB1":0.327,"SUB2":0.324,"SUB3":0.256,"SUB4":0.243,
                 "SUB5":0.246,"SUB6":0.263,"SUB7":0.229,"SUB8":0.304,
                 "SUB9":0.333,"SUB10":0.429}
hybrid_pccs = {"SUB1":0.433,"SUB2":0.424,"SUB3":0.323,"SUB4":0.374,
               "SUB5":0.415,"SUB6":0.450,"SUB7":0.367,"SUB8":0.459,
               "SUB9":0.424,"SUB10":0.457}
for s in ALL_SUBJECTS:
    b = baseline_pccs[s]
    ft = subj_FT.get(s, 0)
    h = hybrid_pccs[s]
    print(f"{s:<8} {b:>10.3f} {ft:>10.3f} {h:>10.3f} {ft-b:>+8.3f} {h-b:>+10.3f}")


print("\n\n✅ ALL EXPERIMENTS COMPLETE!")
print("📸 Screenshot this entire output and send it back.")



📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊
EXPERIMENT 1: PARAMETER COUNTS
📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊

  Baseline CNN-BiLSTM:         1,962,128 params
  Full Hybrid (5 blocks):      7,100,304 params
    — Transferred from base:   1,941,648
    — New (MR-ResBlocks etc):   5,158,656
  Conv-Only (3 blocks):        4,624,016 params
  No Post-LSTM (4 blocks):     5,806,736 params
  No Pre-LSTM (4 blocks):      5,917,584 params

  📋 SUMMARY:
  Model                                Params      PCC
  -------------------------------------------------------
  Baseline                          1,962,128   0.2878
  Fine-tuned Baseline               1,962,128    ?????
  Conv-Only (3 blocks)              4,624,016    ?????
  No Post-LSTM (4 blocks)           5,806,736   0.2814
  No Pre-LSTM (4 blocks)            5,917,584   0.1894
  Full Hybrid (5 blocks)            7,100,304   0.3998


🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴
EXPERIMENT 2: FINE-TUNED BASELINE (CRITICAL!)
Same Phase 2 protocol, NO MR-ResBlocks
🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴